# 🎙️ VAANI AI — Multilingual Speech-to-Text Colab Engine

**Zero-Cost Architecture**: This notebook runs on **Google Colab Free Tier (T4 GPU)**.
- 🚫 ₹0 Budget (No paid OpenAI API, no paid GPU, no paid cloud)
- 💻 Zero load on your local PC (Intel i3 / 8GB RAM remains fast and cool)
- 🌐 Supports: Hindi (Devanagari), English, Gujarati, Punjabi, Bengali, Marathi, Tamil, Telugu, and 90+ languages.
- ⚡ Powered by **Faster-Whisper (CTranslate2)** with **Silero VAD Filtering** for silence removal and 4x faster execution.

### Step 1: Check Runtime Hardware
Colab provides a free T4 GPU. In the top menu, go to **Runtime > Change runtime type** and select **T4 GPU**.

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies
Installs `faster-whisper`, `ffmpeg-python`, and ensures `ffmpeg` is ready.

In [ ]:
!pip install -q faster-whisper requests tqdm
!apt-get install -y ffmpeg
print("✅ All dependencies installed successfully!")

### Step 3: Configure Parameters and Input Video
Set the Job ID and choose how to provide the video (Upload directly or from URL).

In [ ]:
import os, json, subprocess
from google.colab import files

# ⚙️ CONFIGURATION
JOB_ID = "job_sample_01"         # Enter Job ID from VAANI AI Dashboard
MODEL_SIZE = "small"             # Options: 'tiny', 'base', 'small', 'medium' ('small' recommended for balance)
TARGET_LANGUAGE = "auto"         # 'auto' or 'hi' (Hindi), 'en', 'gu', 'bn', 'pa', etc.

print(f"📌 Current Settings: Job ID={JOB_ID}, Model={MODEL_SIZE}, Target Language={TARGET_LANGUAGE}")

### Step 4: Upload Video File
Click the upload button below to select your video (`.mp4`, `.mov`, `.mkv`, `.avi`, `.webm`).

In [ ]:
uploaded = files.upload()
if uploaded:
    INPUT_VIDEO_PATH = list(uploaded.keys())[0]
    print(f"✅ Video uploaded: {INPUT_VIDEO_PATH} ({os.path.getsize(INPUT_VIDEO_PATH) / (1024*1024):.2f} MB)")
else:
    raise ValueError("Please upload a video file to proceed.")

### Step 5: Run AI Transcription (Faster-Whisper + VAD)
Extracts audio, detects language, and performs multilingual transcription with precise timestamps.

In [ ]:
import torch
from faster_whisper import WhisperModel

# 1. Hardware Selection
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"🚀 Engine Running on: {device.upper()} (Compute: {compute_type})")

# 2. Audio Extraction via FFmpeg
AUDIO_PATH = f"extracted_{JOB_ID}.wav"
cmd = ["ffmpeg", "-y", "-i", INPUT_VIDEO_PATH, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", AUDIO_PATH]
subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
print(f"🎵 Audio extracted: {AUDIO_PATH}")

# 3. Load Model
print(f"⏳ Loading Faster-Whisper model '{MODEL_SIZE}'...")
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

# 4. Transcribe with Silero VAD
print("🎙️ Transcribing and detecting language...")
lang_param = None if (not TARGET_LANGUAGE or TARGET_LANGUAGE == "auto") else TARGET_LANGUAGE
segments_gen, info = model.transcribe(
    AUDIO_PATH,
    beam_size=5,
    language=lang_param,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
)

detected_lang = info.language
lang_prob = round(info.language_probability, 4)
duration = round(info.duration, 2)
print(f"🌟 Language Detected: '{detected_lang.upper()}' (Confidence: {lang_prob * 100:.1f}%), Duration: {duration}s")

def format_srt_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{min(millis, 999):03d}"

segments = []
srt_lines = []
full_texts = []

print("\n--- TRANSCRIPT PREVIEW ---")
for idx, seg in enumerate(segments_gen):
    clean_text = seg.text.strip()
    seg_obj = {
        "id": idx,
        "start": round(seg.start, 2),
        "end": round(seg.end, 2),
        "text": clean_text
    }
    segments.append(seg_obj)
    full_texts.append(clean_text)
    srt_lines.append(f"{idx + 1}\n{format_srt_time(seg.start)} --> {format_srt_time(seg.end)}\n{clean_text}\n")
    print(f"[{seg.start:.1f}s - {seg.end:.1f}s]: {clean_text}")

full_text = " ".join(full_texts)
result_payload = {
    "job_id": JOB_ID,
    "detected_language": detected_lang,
    "language_probability": lang_prob,
    "duration": duration,
    "text": full_text,
    "segments": segments
}

# Cleanup temporary audio
if os.path.exists(AUDIO_PATH):
    os.remove(AUDIO_PATH)

print(f"\n🎉 Transcription finished! Total segments: {len(segments)}")

### Step 6: Save and Download Result Files
Downloads `vaani_transcript_<job_id>.json`. You can drop this JSON directly into your VAANI AI web app to load subtitles and interactive workspace!

In [ ]:
json_filename = f"vaani_transcript_{JOB_ID}.json"
srt_filename = f"vaani_transcript_{JOB_ID}.srt"
txt_filename = f"vaani_transcript_{JOB_ID}.txt"

with open(json_filename, "w", encoding="utf-8") as f:
    json.dump(result_payload, f, ensure_ascii=False, indent=2)

with open(srt_filename, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

with open(txt_filename, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"💾 Files written: {json_filename}, {srt_filename}, {txt_filename}")
files.download(json_filename)
print(f"📥 Downloading {json_filename} to your PC...")

### Step 7 (Optional): 1-Click Direct Webhook Sync
If you are using a tunnel (e.g. ngrok or localtunnel) or network address for Django, you can push results back instantly without manual file download.

In [ ]:
import requests

# Enter your Django backend URL if accessible (leave blank to skip)
DJANGO_BACKEND_URL = ""  # e.g., "https://your-tunnel.ngrok-free.app"

if DJANGO_BACKEND_URL:
    webhook_endpoint = f"{DJANGO_BACKEND_URL.rstrip('/')}/api/transcriptions/{JOB_ID}/webhook/"
    print(f"POSTing transcript to {webhook_endpoint}...")
    try:
        resp = requests.post(webhook_endpoint, json=result_payload, timeout=30)
        if resp.status_code == 200:
            print("✅ Synchronized directly with VAANI AI web app!")
        else:
            print(f"⚠️ Server returned status {resp.status_code}: {resp.text}")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
else:
    print("ℹ️ No DJANGO_BACKEND_URL set. Use the downloaded JSON file in the VAANI AI web interface.")